In [ ]:
%load_ext jupyter_black

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import os
from dotenv import load_dotenv

import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

In [ ]:
client = QdrantClient("localhost", port=6333)

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
model_name = "Qwen/Qwen3-Embedding-0.6B"
model = SentenceTransformer(model_name, token=hf_token)

In [ ]:
if not client.collection_exists("legal_docs"):
    client.create_collection(
        collection_name="legal_docs",
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )

In [ ]:
chunks = pd.read_json("../data/processed/chunks/trafik_kanunu.jsonl", lines=True)

In [ ]:
embeddings = model.encode([c for c in chunks["text"].tolist()])
points = [
    PointStruct(id=i, vector=emb.tolist(), payload=chunk)
    for i, (chunk, emb) in enumerate(zip(chunks["metadata"].tolist(), embeddings))
]
client.upsert(collection_name="legal_docs", points=points)